# **1. Mounting Google Drive**

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **2. Install Dependencies for PySpark**

This section installs the required system and Python dependencies for Apache PySpark, including Java (OpenJDK 17), PySpark, and findspark, to enable proper Spark initialization in the notebook environment.

In [3]:
!apt-get update -qq
!apt-get install -y openjdk-17-jdk-headless -qq > /dev/null
!pip install -q -U "pyspark[connect]~=4.0.0" findspark

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


# **3. SparkSession Initialisation**

This section initializes the SparkSession, the main entry point for Spark, and sets key configurations such as JAVA_HOME, driver memory, and shuffle partitions for efficient local execution.

In [4]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = (
    SparkSession.builder
    .appName("AmazonAutomotiveSentiment")
    .master("local[*]")
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "8g")
    .config("spark.sql.shuffle.partitions", "200")
    .getOrCreate()
)

spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark

# **4. Download Dataset**

This section downloads the Amazon Automotive JSONL dataset from Hugging Face and saves it locally for efficient loading and processing with Spark.

In [5]:
!wget -q --show-progress \
    "https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/review_categories/Automotive.jsonl" \
    -O /content/Automotive.jsonl

print("Download complete.")

/content/Automotive 100%[===================>]   8.13G  74.2MB/s    in 90s     
Download complete.


# **5. Data Ingestion with Partition Count**

I loaded the JSON file into a Spark DataFrame and checked its partition count alongside the row and column counts.

In [6]:
df = spark.read.json("/content/Automotive.jsonl")

print(f"Partition count : {df.rdd.getNumPartitions()}")
print(f"Row count       : {df.count():,}")
print(f"Column count    : {len(df.columns)}")

Partition count : 66
Row count       : 19,955,450
Column count    : 10


# **6. Missing Value Handling**

This section checks every column for null values. All columns returned zero missing values, confirming the dataset is already clean and complete with no nulls to handle.

In [7]:
from pyspark.sql.functions import col, count, when

df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

+----+------------+------+-----------+------+----+---------+-----+-------+-----------------+
|asin|helpful_vote|images|parent_asin|rating|text|timestamp|title|user_id|verified_purchase|
+----+------------+------+-----------+------+----+---------+-----+-------+-----------------+
|   0|           0|     0|          0|     0|   0|        0|    0|      0|                0|
+----+------------+------+-----------+------+----+---------+-----+-------+-----------------+



# **7. Removing Duplicate Values**

I checked for duplicate reviews based on user, product, and review text combined. This ensures the same review isn't counted multiple times in my analysis.

In [8]:
# Checking for duplicate reviews based on user, product and review text
before_dedup = df.count()

df = df.dropDuplicates(["user_id", "asin", "text", "timestamp"])

after_dedup = df.count()

print(f"Duplicates removed: {before_dedup - after_dedup:,}")

Duplicates removed: 232,224


# **8. Data Type Validation**

I confirmed that key columns hold the correct data types before running any transformations. This prevents silent errors later in the pipeline if a type mismatch were to occur.

In [9]:
# Confirms key columns hold the correct data types before transformation
# since incorrect types would cause error in the pipeline
print("Data type validation:")
for col_name, expected_type in [
    ("rating", "DoubleType"),
    ("verified_purchase", "BooleanType"),
    ("helpful_vote", "LongType"),
]:
    actual = dict(df.dtypes)[col_name]
    print(f"{col_name:<20} : {actual}")

Data type validation:
rating               : double
verified_purchase    : boolean
helpful_vote         : bigint


# **9. Label Creation**

I converted star ratings into a binary sentiment label, with 4-5 stars as Positive and 1-3 stars as Negative. That leaves 19,723,226 labelled rows, split roughly 78% Positive to 22% Negative.

In [10]:
from pyspark.sql.functions import col, when, count

# Converting star rating into binary sentiment
# 4-5 stars = Positive (1.0), 1-3 stars = Negative (0.0)
automotive_df = df.withColumn("sentiment",
    when(col("rating") >= 4.0, 1.0).otherwise(0.0)
)

# Removing rows where sentiment or text is missing
automotive_df = automotive_df.filter(
    col("sentiment").isNotNull() &
    col("text").isNotNull()
)

print(f"Rows ready for ML: {automotive_df.count():,}")
automotive_df.groupBy("sentiment").count().orderBy("sentiment").show()

Rows ready for ML: 19,723,226
+---------+--------+
|sentiment|   count|
+---------+--------+
|      0.0| 4283459|
|      1.0|15439767|
+---------+--------+



# **10. Text Cleaning**

This section creates a custom function to clean both the review text and title removing HTML tags, stripping special characters, and converting everything to lowercase.

In [11]:
import re
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

# Custom function to clean raw review text before tokenisation
@udf(StringType())
def clean_text(t):
    if t is None: return ""
    t = re.sub(r"<.*?>", "", t)           # remove HTML tags
    t = re.sub(r"[^a-zA-Z0-9 ]", " ", t) # remove special characters
    return t.lower().strip()              # lowercase and trim whitespace

# Applying cleaning to both text and title columns
automotive_df = automotive_df.withColumn("clean_text", clean_text(col("text")))
automotive_df = automotive_df.withColumn("clean_title", clean_text(col("title")))

automotive_df.select("text", "clean_text").show(5, truncate=80)

+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+
|                                                                            text|                                                                      clean_text|
+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+
|2nd day update:<br />See last 2 pictures. It rained and blew a ton of dust th...|2nd day update see last 2 pictures  it rained and blew a ton of dust the nigh...|
|                 These devices broke off of my car the first month I use3d them.|                  these devices broke off of my car the first month i use3d them|
|                      These headlights were easy to install and are very bright.|                       these headlights were easy to install and are very bright|
|Great pump that

# **11. Removing Empty Reviews**

I checked how many reviews became completely empty after cleaning. These are rows where the title and text contained only symbols, emojis, or numbers that got removed. There were 15,994 of them.

An empty review gives the tokeniser nothing to work with and would only add noise to the TF-IDF vectors, so I dropped them. That leaves 19,707,232 rows going forward.

In [12]:
# Checking how many rows have empty cleaned text (title + text combined)
combined_check = F.concat_ws(" ", col("clean_title"), col("clean_text"))
empty_count = automotive_df.filter(F.trim(combined_check) == "").count()
total_count = automotive_df.count()

print(f"Empty after cleaning : {empty_count:,}")
print(f"Total rows           : {total_count:,}")
print(f"Percentage empty     : {(empty_count/total_count)*100:.2f}%")

# Removing rows with no usable text
automotive_df = automotive_df.filter(F.trim(combined_check) != "")
print(f"Rows after removing empty text: {automotive_df.count():,}")

Empty after cleaning : 15,994
Total rows           : 19,723,226
Percentage empty     : 0.08%
Rows after removing empty text: 19,707,232


# **12. Feature Engineering**

I engineered five new features: review and title length, a flag for detailed reviews over 200 characters, helpful_vote cast to integer, and verified_purchase encoded as 1 or 0. I also combined the cleaned title and text into a single review_combined column for TF-IDF, leaving 19,707,232 ML-ready rows.

In [13]:
from pyspark.sql.functions import length

# Adding review length as a numeric feature
automotive_df = automotive_df.withColumn(
    "review_length", length(col("clean_text"))
)

# Adding title length as a numeric feature
automotive_df = automotive_df.withColumn(
    "title_length", length(col("clean_title"))
)

# Flagging reviews longer than 200 characters as detailed reviews
automotive_df = automotive_df.withColumn(
    "is_long_review", when(length(col("clean_text")) > 200, 1.0).otherwise(0.0)
)

# Casting helpful_vote to integer
automotive_df = automotive_df.withColumn(
    "helpful_vote", col("helpful_vote").cast("int")
)

# Encoding verified_purchase boolean to integer (True=1, False=0)
automotive_df = automotive_df.withColumn(
    "verified_int", when(col("verified_purchase") == True, 1.0).otherwise(0.0)
)

# Combining clean_title and clean_text into one column for TF-IDF
automotive_df = automotive_df.withColumn(
    "review_combined", F.concat_ws(" ", col("clean_title"), col("clean_text"))
)

# Selecting only the columns needed for the pipeline
automotive_df = automotive_df.select(
    "review_combined",
    "review_length",
    "title_length",
    "is_long_review",
    "helpful_vote",
    "verified_int",
    "sentiment"
)

print(f"Features ready: {automotive_df.count():,}")
automotive_df.show(5, truncate=80)

Features ready: 19,707,232
+--------------------------------------------------------------------------------+-------------+------------+--------------+------------+------------+---------+
|                                                                 review_combined|review_length|title_length|is_long_review|helpful_vote|verified_int|sentiment|
+--------------------------------------------------------------------------------+-------------+------------+--------------+------------+------------+---------+
|covers the jeep without a lot of fuss we love to take the top and doors off f...|          436|          37|           1.0|           5|         1.0|      1.0|
|these are amazing  just replaced my signal bulbs with leds a these are amazin...|          423|          60|           1.0|           0|         1.0|      1.0|
|not anti theft i had my license plate stolen so i ordered these because the t...|          461|          14|           1.0|           1|         1.0|      0.0|
|five s

# **13. Summary Statistics**

I reviewed the numeric ranges of my engineered features before scaling them. This confirms why StandardScaler is needed, since helpful_vote and review_length operate on very different scales.

In [14]:
# Reviewing numeric ranges before scaling justifies why StandardScaler
# is needed — helpful_vote and review_length have very different scales
automotive_df.select("review_length", "title_length", "helpful_vote").describe().show()

+-------+------------------+------------------+------------------+
|summary|     review_length|      title_length|      helpful_vote|
+-------+------------------+------------------+------------------+
|  count|          19707232|          19707232|          19707232|
|   mean|  155.576331521342|21.430759936250816|0.6515641567522015|
| stddev|230.80274633964578|16.441558337183636| 5.663166460227792|
|    min|                 0|                 0|                -4|
|    max|             28963|               462|              2822|
+-------+------------------+------------------+------------------+



# **14. Pattern Analysis — Review Length by Sentiment**

I checked whether review length correlates with sentiment class. This justifies keeping review_length as a meaningful engineered feature going into the pipeline.

In [15]:
# Checking whether review length correlates with sentiment justifies
# keeping review_length as an engineered feature
automotive_df.groupBy("sentiment").agg(
    F.round(F.avg("review_length"), 1).alias("avg_review_length"),
    F.round(F.avg("helpful_vote"), 2).alias("avg_helpful_votes")
).orderBy("sentiment").show()

+---------+-----------------+-----------------+
|sentiment|avg_review_length|avg_helpful_votes|
+---------+-----------------+-----------------+
|      0.0|            204.4|             0.88|
|      1.0|            142.0|             0.59|
+---------+-----------------+-----------------+



# **15. TF-IDF Pipeline**

I built the first four pipeline stages: RegexTokenizer splits the combined review text into individual words, StopWordsRemover filters out common filler words, HashingTF converts the remaining words into term frequency vectors, and IDF downweights words that appear in too many reviews to be useful.

In [16]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    RegexTokenizer,
    StopWordsRemover,
    HashingTF,
    IDF,
    VectorAssembler,
    StandardScaler
)

# Stage 1: RegexTokenizer — split review text into individual words
tokenizer = RegexTokenizer(
    inputCol="review_combined",
    outputCol="tokens",
    pattern="\\W+",
    minTokenLength=2
)

# Stage 2: StopWordsRemover — remove common words that add no value
remover = StopWordsRemover(
    inputCol="tokens",
    outputCol="filtered"
)

# Stage 3: HashingTF — convert filtered words into term frequency vectors
htf = HashingTF(
    inputCol="filtered",
    outputCol="raw_tfidf",
    numFeatures=65536
)

# Stage 4: IDF — downweight common terms across all reviews
idf = IDF(
    inputCol="raw_tfidf",
    outputCol="tfidf_features",
    minDocFreq=5
)

# **16. Encoding and Scaling**

I added the final two stages: VectorAssembler combines the TF-IDF text features with the numeric features into a single vector, and StandardScaler normalises everything to a comparable scale. All 6 stages are then assembled into the complete pipeline.

In [17]:
# Stage 5: VectorAssembler — combine TF-IDF features with numeric features
assembler = VectorAssembler(
    inputCols=["tfidf_features", "review_length", "title_length", "is_long_review", "helpful_vote", "verified_int"],
    outputCol="raw_features"
)

# Stage 6: StandardScaler — normalise all features to the same scale
scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="features",
    withStd=True,
    withMean=False
)

# Building the pipeline with all 6 stages in order
pipeline = Pipeline(stages=[tokenizer, remover, htf, idf, assembler, scaler])

print(f"Pipeline constructed successfully with {len(pipeline.getStages())} stages")

Pipeline constructed successfully with 6 stages


# **17. Train/Test Split and Pipeline Fit**

I split the data into 80% training and 20% testing, then fit the pipeline on the training set and transformed both splits.

In [18]:
import time

# Splitting into 80% training and 20% testing
training_data, testing_data = automotive_df.randomSplit([0.8, 0.2], seed=42)

training_data.cache()
testing_data.cache()

# Fitting pipeline and transforming both splits
start_time = time.time()

auto_pipeline_model = pipeline.fit(training_data)

training_output = auto_pipeline_model.transform(training_data).cache()
testing_output  = auto_pipeline_model.transform(testing_data).cache()

print(f"Training rows : {training_output.count():,}")
print(f"Testing rows  : {testing_output.count():,}")
print(f"Pipeline completed in : {time.time()-start_time:.1f}s")

training_output.select("sentiment", "features").show(15, truncate=80)

Training rows : 15,763,810
Testing rows  : 3,943,422
Pipeline completed in : 5633.3s
+---------+--------------------------------------------------------------------------------+
|sentiment|                                                                        features|
+---------+--------------------------------------------------------------------------------+
|      1.0|(65541,[5809,7642,26479,32888,35665,40091,44331,44873,46045,46375,46773,56734...|
|      1.0|(65541,[1903,4529,25656,31144,32558,33181,34156,34606,37756,38742,46128,55853...|
|      1.0|(65541,[48821,62382,65536,65539,65540],[23.13985279289581,23.287828798853585,...|
|      0.0|(65541,[9761,12215,12498,20006,24707,32697,32814,33643,34498,34717,35209,3628...|
|      1.0|(65541,[34064,52879,56774,65262,65536,65540],[57.14299577304393,1.98650075402...|
|      1.0|(65541,[120,7491,11650,62537,65536,65540],[17.624686573506633,20.278346810295...|
|      1.0|(65541,[46990,65536,65540],[18.832650375790273,0.021663952342854707

# **18. Selecting Final Columns and Saving Processed Data as Parquet**

In [19]:
training_final = training_output.select("sentiment", "features")
testing_final  = testing_output.select("sentiment", "features")

output_dir = "/content/drive/MyDrive/amazon_automotive_sentiment"

training_final.write.mode("overwrite").parquet(f"{output_dir}/training_data.parquet")
testing_final.write.mode("overwrite").parquet(f"{output_dir}/testing_data.parquet")

print("Parquet files saved to Google Drive:")
print(f"{output_dir}/training_data.parquet")
print(f"{output_dir}/testing_data.parquet")

Parquet files saved to Google Drive:
/content/drive/MyDrive/amazon_automotive_sentiment/training_data.parquet
/content/drive/MyDrive/amazon_automotive_sentiment/testing_data.parquet


# **19. Verifying Saved Parquet**

In [20]:
training_verify = spark.read.parquet(f"{output_dir}/training_data.parquet")
testing_verify  = spark.read.parquet(f"{output_dir}/testing_data.parquet")

print("Parquet verification:")
print(f"Trained rows : {training_verify.count():,}")
print(f"Tested rows  : {testing_verify.count():,}")

training_verify.printSchema()

Parquet verification:
Trained rows : 15,763,810
Tested rows  : 3,943,422
root
 |-- sentiment: double (nullable = true)
 |-- features: vector (nullable = true)



# **20. Spark Session Termination**
This section terminates the active SparkSession to release allocated memory and cluster resources. Properly closing the session prevents resource leaks and ensures a clean environment for future runs.

In [21]:
spark.stop()
print("Spark session stopped successfully.")

Spark session stopped successfully.
